# Day 07：第一周复习与周测（Jupyter Notebook 实战手册）

> **所属阶段**：第 1 阶段 · 第一阶段：Python 编程基础 ｜ **第 1 周**：第 1 周：Python 核心语法与数据结构基石  
> **GitLab 工单**：`STARMIND-007` ｜ **开发分支**：`feat/day07-week1-contacts-project`  
> **配套 HTML5 课件**：`courseware/day07/index.html` ｜ **独立版本快照**：`versions/day07/`

---

## 🎙️ 老陈开场旁白

眨眼第一周就到收官日了！今天不塞新语法，咱们干两件检验真功夫的事：上午先做第一周知识点串讲 + 自动化周测题库自检，查漏补缺，看看哪些坑还容易踩；下午直接进入第一个完整闭环小项目 STARMIND-007——「星辰数科企业通讯录管理系统（支持增删改查、模糊搜索、按部门统计、手机号脱敏以及 JSON 文件持久化）」。阿杰今天会亲自跑全套集成测试，通过了咱们第一周就圆满通关！

### 承上启下
- **昨日基线**：前六天咱们从环境搭建一路打到变量、字符串、流程控制、list/tuple/set/dict、JSON 和函数管道，第一周地基已全部铺完。
- **明日预告**：下周（Day 8）进入第二周，咱们正式开启面向对象编程（OOP），把零散的字典升级成带校验和行为的 `ChatMessage` 核心类！


In [ ]:
# 环境自检与路径初始化：确保可以无缝导入当日版本模块并支持异步协程运行
import os
import sys
from pathlib import Path

# 允许在 Jupyter 已有事件循环内安全调用 asyncio.run()
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'versions').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DAY_DIR = PROJECT_ROOT / 'versions' / 'day07'
for _p in (DAY_DIR, DAY_DIR / 'src', PROJECT_ROOT):
    if _p.exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
os.chdir(DAY_DIR)
__file__ = str(DAY_DIR / 'src' / 'notebook_entry.py')

print(f'✅ Python 版本: {sys.version.split()[0]}')
print(f'✅ 项目根目录: {PROJECT_ROOT}')
print(f'✅ 当日版本目录: {DAY_DIR}')


## 📋 企业需求背景与验收目标（STARMIND-007）

HR 与行政部门需要一个离线可用的内部通讯录 CLI 系统，支持快速录入、修改、删除员工联系方式，并持久化保存到本地 JSON 文件，便于星辰智脑后续接入查人技能。

**今日迭代目标：**
- 完成第一周核心知识（类型/切片/流程/容器/JSON/函数）综合自测与复盘。
- 独立实现通讯录完整 CRUD、关键字模糊搜索（姓名/部门/标签）、JSON 加载与保存。
- 确保所有业务操作与异常边界通过自动化集成测试。

**验收标准：**
- 支持按工号防重添加、更新、删除、精确查询与关键词模糊搜索。
- 重启后能从 JSON 文件完整恢复所有联系人数据，中文无转义乱码。
- 内置第一周自测判分引擎可自动评估答题正确率。

---

## 📚 课堂核心技术笔记

### 1. 1. 第一周高频易错点清扫清单

1）`bool('False')` 为 `True`；2）字符串不可变，方法调用必须重新赋值；3）遍历列表禁止原地 `remove`；4）空集合是 `set()` 不是 `{}`；5）`json.dumps` 必加 `ensure_ascii=False`；6）函数默认参数严禁写 `[]` 或 `{}`。

> **⚠️ 老陈避坑**：每次写完代码养成跑一遍 `pytest` 的习惯，能提前拦截 80% 的低级拼写与边界错误。

### 2. 2. 主键索引设计：为什么通讯录用 dict 而不是 list？

用 `emp_id` 作键的字典 `{emp_id: {name, dept, phone, ...}}`，按工号查人、改人、删人全是 O(1) 瞬间完成，且天然保证工号唯一不重复。

> **⚠️ 老陈避坑**：注意 JSON 的键只能是字符串！即使你在内存里用整数 `1001` 当字典 key，经过 `json.dump` 再 `json.load` 回来后 key 会变成字符串 `'1001'`，所以统一把 `emp_id` 定义为 `str`（如 `'XM-0001'`）是最省心的工程实践。

### 3. 3. 持久化原子写入思路

生产级本地 JSON 持久化通常遵循：读取时若文件不存在返回空库 `{}`；写入时先确保父目录存在（`mkdir(parents=True, exist_ok=True)`），再以 UTF-8 覆盖写出。

> **⚠️ 老陈避坑**：如果文件内容被人为改坏导致 `JSONDecodeError`，应在加载层给出明确错误提示，避免静默覆盖把旧数据清空。


## 💻 实操步骤 1：步骤一：实现第一周知识点自动化周测判分引擎

- **目标文件**：`versions/day07/src/day07_week1_quiz.py`
- **老陈代码旁白**：先用 10 分钟跑一遍咱们第一周的 5 道核心陷阱周测题，全部答对拿 100 分再做大项目！


In [ ]:
"""
STARMIND-007 模块一：第一周 Python 核心语法自动化周测判分引擎。
涵盖类型转换、切片、集合运算、JSON 参数与默认参数陷阱五大高频考点。
"""

from typing import Dict, List, Any

# 周测标准题库（题号、题干、标准答案、解析）
WEEK1_QUESTIONS: List[Dict[str, str]] = [
    {
        "id": "Q1",
        "topic": "布尔类型转换",
        "question": "表达式 bool('False') 的求值结果是 True 还是 False？",
        "answer": "True",
        "explanation": "非空字符串转 bool 永远为 True，只有空字符串 '' 为 False。",
    },
    {
        "id": "Q2",
        "topic": "空集合创建",
        "question": "在 Python 中创建一个空集合应该写 set() 还是 {}？",
        "answer": "set()",
        "explanation": "{} 创建的是空字典 dict，空集合必须写 set()。",
    },
    {
        "id": "Q3",
        "topic": "字符串切片",
        "question": "表达式 'StarMind'[0:4] 的结果是什么？",
        "answer": "Star",
        "explanation": "切片遵循左闭右开原则，取索引 0, 1, 2, 3 四个字符。",
    },
    {
        "id": "Q4",
        "topic": "JSON 中文输出",
        "question": "json.dumps 中防止中文被转义为 \\uXXXX 的关键参数名是什么？",
        "answer": "ensure_ascii",
        "explanation": "设置 ensure_ascii=False 即可保留可读中文字符。",
    },
    {
        "id": "Q5",
        "topic": "函数默认参数",
        "question": "需要列表型可选参数时，函数签名中默认值应写 None 还是 []？",
        "answer": "None",
        "explanation": "可变默认参数 [] 会在多次调用间共享导致数据串台，必须用 None 占位。",
    },
]


def grade_quiz(submissions: Dict[str, str]) -> Dict[str, Any]:
    """根据学员提交的答案字典自动判分并生成错题诊断报告。"""
    total = len(WEEK1_QUESTIONS)
    correct = 0
    wrong_details: List[Dict[str, str]] = []

    for q in WEEK1_QUESTIONS:
        qid = q["id"]
        user_ans = str(submissions.get(qid, "")).strip()
        if user_ans == q["answer"]:
            correct += 1
        else:
            wrong_details.append(
                {
                    "id": qid,
                    "topic": q["topic"],
                    "your_answer": user_ans or "(未作答)",
                    "expected": q["answer"],
                    "explanation": q["explanation"],
                }
            )

    score = int(round((correct / total) * 100)) if total else 0
    return {
        "total": total,
        "correct": correct,
        "score": score,
        "passed": score >= 80,
        "wrong_details": wrong_details,
    }


if __name__ == "__main__":
    perfect_answers = {"Q1": "True", "Q2": "set()", "Q3": "Star", "Q4": "ensure_ascii", "Q5": "None"}
    report = grade_quiz(perfect_answers)
    print(f"第一周周测成绩: {report['score']} 分 | 是否通过: {report['passed']}")


### 🔍 步骤 1 代码深度解析

通过把周测题库结构化为列表字典，既能直接运行判分，也能直观复盘第一周最容易踩的 5 个坑。

**预期运行输出参考：**
```text
第一周周测成绩: 100 分 | 是否通过: True
```


## 💻 实操步骤 2：步骤二：实现星辰企业通讯录管理系统（CRUD + 模糊搜索 + JSON 持久化）

- **目标文件**：`versions/day07/src/day07_contacts_app.py`
- **老陈代码旁白**：重头戏来了！把第一周学的所有技能揉在一起，交付完整的企业通讯录管理系统，支持持久化到 JSON 并按部门统计。


In [ ]:
"""
STARMIND-007 模块二：星辰数科企业通讯录管理系统（含 JSON 持久化）。
综合运用 dict 主键索引、字符串清洗脱敏、list/set 标签、函数封装与 JSON 文件持久化。
"""

import json
from pathlib import Path
from typing import Dict, Any, List, Optional

# 通讯录内存数据结构：{emp_id: {"emp_id": ..., "name": ..., "dept": ..., "phone": ..., "skills": [...]}}
ContactBook = Dict[str, Dict[str, Any]]


def load_contacts(db_path: Path) -> ContactBook:
    """从 JSON 文件加载通讯录；文件不存在时返回空字典。"""
    if not db_path.exists():
        return {}
    with db_path.open("r", encoding="utf-8") as f:
        data = json.load(f)
    return data if isinstance(data, dict) else {}


def save_contacts(db_path: Path, book: ContactBook) -> Path:
    """将通讯录持久化写入 UTF-8 JSON 文件。"""
    db_path.parent.mkdir(parents=True, exist_ok=True)
    with db_path.open("w", encoding="utf-8") as f:
        json.dump(book, f, ensure_ascii=False, indent=2)
    return db_path


def mask_phone(phone: str) -> str:
    """手机号中间四位脱敏，便于展示层安全输出。"""
    p = phone.strip()
    return f"{p[:3]}****{p[-4:]}" if len(p) == 11 and p.isdigit() else "已隐藏"


def upsert_contact(
    book: ContactBook,
    emp_id: str,
    name: str,
    dept: str,
    phone: str,
    skills: Optional[List[str]] = None,
) -> Dict[str, Any]:
    """新增或更新员工联系人记录，并自动对技能标签去重排序。"""
    eid = emp_id.strip().upper()
    if not eid or not name.strip():
        raise ValueError("工号与姓名不能为空")

    clean_skills = sorted({s.strip().lower() for s in (skills or []) if s.strip()})
    record: Dict[str, Any] = {
        "emp_id": eid,
        "name": name.strip(),
        "dept": dept.strip(),
        "phone": phone.strip(),
        "masked_phone": mask_phone(phone),
        "skills": clean_skills,
    }
    book[eid] = record
    return record


def delete_contact(book: ContactBook, emp_id: str) -> bool:
    """按工号删除联系人，存在并删除返回 True，否则返回 False。"""
    eid = emp_id.strip().upper()
    return book.pop(eid, None) is not None


def search_contacts(book: ContactBook, keyword: str) -> List[Dict[str, Any]]:
    """按姓名、部门、工号或技能标签执行大小写不敏感模糊搜索。"""
    kw = keyword.strip().lower()
    if not kw:
        return sorted(book.values(), key=lambda c: c["emp_id"])

    matched: List[Dict[str, Any]] = []
    for contact in book.values():
        haystack = " ".join(
            [
                contact["emp_id"],
                contact["name"],
                contact["dept"],
                " ".join(contact.get("skills", [])),
            ]
        ).lower()
        if kw in haystack:
            matched.append(contact)
    return sorted(matched, key=lambda c: c["emp_id"])


def summarize_by_dept(book: ContactBook) -> Dict[str, int]:
    """按部门统计通讯录人数。"""
    summary: Dict[str, int] = {}
    for contact in book.values():
        dept = contact["dept"]
        summary[dept] = summary.get(dept, 0) + 1
    return summary


if __name__ == "__main__":
    book: ContactBook = {}
    upsert_contact(book, "XM-0001", "老陈", "AI研发部", "13800001111", ["Python", "架构", "LLM"])
    upsert_contact(book, "XM-0002", "林产品", "产品部", "13900002222", ["PRD", "Prompt"])
    upsert_contact(book, "XM-0003", "阿杰", "测试部", "13700003333", ["Pytest", "Python"])
    print("搜索 'python' 命中:", [c["name"] for c in search_contacts(book, "python")])
    print("部门人数统计:", summarize_by_dept(book))


### 🔍 步骤 2 代码深度解析

1. `upsert_contact` 同时承担「新增」和「更新」（Update + Insert = Upsert），并通过 `eid = emp_id.strip().upper()` 统一主键大小写规范。
2. `search_contacts` 将多个可搜索字段拼成 `haystack` 字符串统一小写匹配，简洁高效，还支持按技能查人！

**预期运行输出参考：**
```text
搜索 'python' 命中: ['老陈', '阿杰']
部门人数统计: {'AI研发部': 1, '产品部': 1, '测试部': 1}
```


## 🧪 当日自动化单元测试

在终端运行 `pytest versions/day07/tests/test_day07.py -v` 即可验收全部测试断言。下面展示完整测试脚本源码，供对照学习测试写法：

```python
"""STARMIND-007 第一周综合验收测试。"""

import sys
from pathlib import Path
import pytest

ROOT = Path(__file__).resolve().parent.parent
for p in (ROOT, ROOT / "src"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from day07_week1_quiz import grade_quiz
from day07_contacts_app import (
    load_contacts,
    save_contacts,
    upsert_contact,
    delete_contact,
    search_contacts,
    summarize_by_dept,
)


def test_week1_quiz_grading():
    res = grade_quiz({"Q1": "True", "Q2": "set()", "Q3": "Star", "Q4": "ensure_ascii", "Q5": "None"})
    assert res["score"] == 100
    assert res["passed"] is True

    partial = grade_quiz({"Q1": "False"})
    assert partial["passed"] is False
    assert len(partial["wrong_details"]) == 5


def test_contacts_full_lifecycle_and_json_persistence(tmp_path: Path):
    db_file = tmp_path / "data" / "contacts.json"
    # 首次加载文件不存在应返回空字典
    book = load_contacts(db_file)
    assert book == {}

    upsert_contact(book, "xm-001", "老陈", "AI研发部", "13857101111", ["Python", "python", "LLM"])
    upsert_contact(book, "xm-002", "阿杰", "测试部", "13957102222", ["Pytest"])
    assert book["XM-001"]["masked_phone"] == "138****1111"
    assert book["XM-001"]["skills"] == ["llm", "python"]

    # 持久化并模拟重启重新加载
    save_contacts(db_file, book)
    reloaded = load_contacts(db_file)
    assert len(reloaded) == 2
    assert len(search_contacts(reloaded, "研发")) == 1
    assert summarize_by_dept(reloaded) == {"AI研发部": 1, "测试部": 1}

    assert delete_contact(reloaded, "XM-002") is True
    assert delete_contact(reloaded, "XM-999") is False

    with pytest.raises(ValueError):
        upsert_contact(reloaded, "", "无名", "部门", "13800000000")

```


## 📝 课后梯度实战作业与标准答案

建议先在下方新建空白单元格独立完成编码，运行通过后再核对参考答案。


### 作业 1（基础巩固）：基础巩固：通讯录导出 CSV 文本行

**题目要求：**
编写 `export_contacts_csv_text(book: dict) -> str`，将通讯录字典转换为带表头 `emp_id,name,dept,masked_phone` 的 CSV 格式字符串（按工号升序）。

**思路提示：** 遍历排序后的联系人，把每行字段用 `','.join(...)` 拼接，再用 `'\n'.join(lines)` 合并。


In [ ]:
"""Day 7 课后题 1：通讯录导出 CSV 文本。"""

from typing import Dict, Any, List


def export_contacts_csv_text(book: Dict[str, Dict[str, Any]]) -> str:
    """将通讯录导出为标准 CSV 字符串（含表头）。"""
    lines: List[str] = ["emp_id,name,dept,masked_phone"]
    for eid in sorted(book.keys()):
        c = book[eid]
        lines.append(f"{c['emp_id']},{c['name']},{c['dept']},{c['masked_phone']}")
    return "\n".join(lines)


if __name__ == "__main__":
    sample = {
        "XM-001": {"emp_id": "XM-001", "name": "老陈", "dept": "AI研发部", "masked_phone": "138****1111"}
    }
    print(export_contacts_csv_text(sample))


**答案解析与评分标准：**

考查字典排序遍历与格式化文本导出，为 Day 11 的标准库 `csv` 模块做铺垫。


### 作业 2（企业实战）：企业实战：批量导入通讯录冲突检测器

**题目要求：**
HR 批量导入一批新员工列表时，需要提前检测哪些工号已经存在于现有通讯录中。编写 `detect_import_conflicts(book: dict, incoming: list[dict]) -> tuple[list[str], list[dict]]`，返回 `(冲突工号列表, 可安全新增列表)`。

**思路提示：** 用 `eid in book` 做 O(1) 冲突判定，注意同一批导入数据内部也可能重复，用 `seen` 集合同步拦截。


In [ ]:
"""Day 7 课后题 2：批量导入通讯录工号冲突检测。"""

from typing import Dict, Any, List, Tuple


def detect_import_conflicts(
    book: Dict[str, Any], incoming: List[Dict[str, Any]]
) -> Tuple[List[str], List[Dict[str, Any]]]:
    """检测批量导入中的重复工号（含库内重复与批次内重复）。"""
    conflicts: List[str] = []
    safe_items: List[Dict[str, Any]] = []
    batch_seen = set()

    for item in incoming:
        eid = str(item.get("emp_id", "")).strip().upper()
        if eid in book or eid in batch_seen:
            conflicts.append(eid)
        else:
            batch_seen.add(eid)
            safe_items.append(item)
    return conflicts, safe_items


if __name__ == "__main__":
    existing = {"XM-001": {"name": "老陈"}}
    batch = [
        {"emp_id": "XM-001", "name": "重复老陈"},
        {"emp_id": "XM-002", "name": "新人A"},
        {"emp_id": "XM-002", "name": "批次内重复新人A"},
    ]
    print(detect_import_conflicts(existing, batch))


**答案解析与评分标准：**

同时检测「与存量库冲突」和「导入批次内部自冲突」，这是企业数据导入功能最容易遗漏的边界条件。


### 作业 3（架构进阶）：架构进阶：带备份快照的安全保存函数

**题目要求：**
直接覆盖写 `contacts.json` 若中途断电可能损坏文件。编写 `safe_save_with_backup(db_path: Path, data: dict) -> Path`，若目标文件已存在则先复制一份 `.bak` 备份文件，再写入新 JSON。

**思路提示：** 使用 `shutil.copy2` 或 `read_text`/`write_text` 生成 `db_path.with_suffix('.json.bak')`。


In [ ]:
"""Day 7 课后题 3：带自动备份快照的 JSON 安全落盘。"""

import json
import tempfile
from pathlib import Path
from typing import Dict, Any


def safe_save_with_backup(db_path: Path, data: Dict[str, Any]) -> Path:
    """若旧文件存在则先生成 .bak 备份，再写入最新 JSON 数据。"""
    db_path.parent.mkdir(parents=True, exist_ok=True)
    if db_path.exists():
        backup_path = db_path.with_suffix(db_path.suffix + ".bak")
        backup_path.write_text(db_path.read_text(encoding="utf-8"), encoding="utf-8")
    with db_path.open("w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    return db_path


if __name__ == "__main__":
    with tempfile.TemporaryDirectory() as tmp:
        p = Path(tmp) / "contacts.json"
        safe_save_with_backup(p, {"v": 1})
        safe_save_with_backup(p, {"v": 2})
        print("主文件:", p.read_text(encoding="utf-8").strip())
        print("备份文件存在:", p.with_suffix(".json.bak").exists())


**答案解析与评分标准：**

利用 `tempfile.TemporaryDirectory()` 做无残留自测，并引入数据持久化防损坏的备份思维。


---

## 🎯 老陈收尾复盘

恭喜你拿下第一周综合项目！从零基础到独立写出带 JSON 持久化、脱敏、模糊搜索和自动化单测的完整通讯录系统，只用了 7 天。周末好好歇半天，下周一咱们开启面向对象编程（OOP），给星辰智脑搭起真正的模型抽象层。
